In [ ]:
# 0 · setup
import subprocess, sys
def pip(*pkgs): subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=False)
pip('scikit-survival')
import os, json, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from sksurv.metrics import concordance_index_censored
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', device)

In [ ]:
# 1 · load bundle
BUNDLE, METAF = 'train_bundle.npz', 'train_bundle_meta.json'
if not (os.path.exists(BUNDLE) and os.path.exists(METAF)):
    from google.colab import files
    print('Select train_bundle.npz AND train_bundle_meta.json:'); files.upload()
z = np.load(BUNDLE, allow_pickle=True); meta = json.load(open(METAF))
N, V, K, L = meta['n_rows'], meta['num_nodes'], meta['n_bins'], meta['hist_len']
assert 'tte_min' in z.files, 'rebuild the bundle with dataset.py (needs point-process targets)'
print('arrays:', {k: tuple(z[k].shape) for k in z.files})
print('rows %d nodes %d bins %d L %d | next-event observed %.1f%%' % (N, V, K, L, 100*z['tte_observed'].mean()))

In [ ]:
# 2 · tensors + normalized adjacency
def T(n, dt): return torch.as_tensor(z[n], dtype=dt).to(device)
X_num=T('X_num',torch.float32); X_cat=T('X_cat',torch.long)
node_id=T('node_id',torch.long); hist_idx=T('hist_idx',torch.long)
node_feat=T('node_feat',torch.float32); edge_index=T('edge_index',torch.long)
bin_idx=T('bin_idx',torch.long); road=T('road_closure',torch.long); prio=T('priority_high',torch.long)
durations=z['durations'].astype('float32'); events=z['events'].astype('int64'); cuts=z['cuts'].astype('float32')
split=z['split']
tte=z['tte_min'].astype('float32'); tte_obs=z['tte_observed'].astype('int64')
dt_h = torch.as_tensor(np.clip(tte/60.0, 1e-3, 720.0), dtype=torch.float32, device=device)  # gap in hours, capped 30d
tte_obs_t = torch.as_tensor(tte_obs, dtype=torch.float32, device=device)
ev_t = torch.as_tensor(events, dtype=torch.float32, device=device)

def norm_adj(ei, V):
    loops=torch.arange(V,device=ei.device); ei=torch.cat([ei,torch.stack([loops,loops])],1)
    src,dst=ei[0],ei[1]
    deg=torch.zeros(V,device=ei.device).scatter_add_(0,dst,torch.ones_like(dst,dtype=torch.float32))
    dinv=deg.pow(-0.5); dinv[torch.isinf(dinv)]=0.0
    return torch.sparse_coo_tensor(ei, dinv[src]*dinv[dst], (V,V)).coalesce()
A=norm_adj(edge_index,V)
tr_idx=torch.where(torch.as_tensor(split==0))[0].to(device)
va_idx=torch.where(torch.as_tensor(split==1))[0].to(device)
te_idx=torch.where(torch.as_tensor(split==2))[0].to(device)
va_t,va_e=durations[split==1],events[split==1]
print('train/val/test', tr_idx.numel(), va_idx.numel(), te_idx.numel())

In [ ]:
# 3 · trunk (GCN+GRU, as Tier 0) + MMoE + heads
class GCN(nn.Module):
    def __init__(s,fin,h,out): super().__init__(); s.l1=nn.Linear(fin,h); s.l2=nn.Linear(h,out)
    def forward(s,x,A): return torch.sparse.mm(A, s.l2(F.relu(torch.sparse.mm(A, s.l1(x)))))

class EventFeat(nn.Module):
    def __init__(s,n_num,cards,emb=8,out=64):
        super().__init__(); s.embs=nn.ModuleList([nn.Embedding(c,emb) for c in cards])
        s.proj=nn.Linear(n_num+emb*len(cards),out)
    def forward(s,xn,xc): return s.proj(torch.cat([xn]+[e(xc[:,i]) for i,e in enumerate(s.embs)],1))

class Trunk(nn.Module):
    def __init__(s,n_num,cards,fnode,d=64,p=0.2):
        super().__init__(); s.d=d; s.feat=EventFeat(n_num,cards,out=d); s.gcn=GCN(fnode,64,d)
        s.gru=nn.GRU(d,d,batch_first=True)
        s.fuse=nn.Sequential(nn.Linear(d*3,d),nn.ReLU(),nn.Dropout(p),nn.Linear(d,d))
    def event_vectors(s,xn,xc): return s.feat(xn,xc)
    def forward(s,idx,all_vec):
        e=all_vec[idx]; g=s.gcn(node_feat,A)[node_id[idx]]
        bank=torch.cat([all_vec,torch.zeros(1,s.d,device=all_vec.device)],0)
        _,hn=s.gru(bank[hist_idx[idx]])
        return s.fuse(torch.cat([e,g,hn.squeeze(0)],1))

class MMoE(nn.Module):
    # shared experts + per-task gates -> one representation per task
    def __init__(s,d,n_experts=4,n_tasks=4):
        super().__init__()
        s.experts=nn.ModuleList([nn.Sequential(nn.Linear(d,d),nn.ReLU()) for _ in range(n_experts)])
        s.gates=nn.ModuleList([nn.Linear(d,n_experts) for _ in range(n_tasks)])
    def forward(s,H):
        E=torch.stack([ex(H) for ex in s.experts],1)            # [B,n_exp,d]
        return [ (F.softmax(g(H),-1).unsqueeze(-1)*E).sum(1) for g in s.gates ]

class Heads(nn.Module):
    def __init__(s,d,K):
        super().__init__(); s.dur=nn.Linear(d,K); s.pp=nn.Linear(d,1); s.clo=nn.Linear(d,1); s.pri=nn.Linear(d,1)
    def forward(s,reps):
        lam=F.softplus(s.pp(reps[1]).squeeze(-1))+1e-4          # positive intensity (per hour)
        return s.dur(reps[0]), lam, s.clo(reps[2]).squeeze(-1), s.pri(reps[3]).squeeze(-1)

In [ ]:
# 4 · losses: DeepHit (survival) + exponential point-process (Hawkes) + aux
def deephit_nll(logp, bins, ev):
    pmf=logp.exp(); idx=bins.clamp(0,pmf.shape[1]-1)[:,None]
    lp=logp.gather(1,idx).squeeze(1)
    tail=torch.flip(torch.cumsum(torch.flip(pmf,[1]),1),[1]).gather(1,idx).squeeze(1).clamp_min(1e-8).log()
    return -(ev*lp+(1-ev)*tail).mean()

def rank_loss(pmf, bins, ev, sigma=0.1, npairs=4096):
    cif=torch.cumsum(pmf,1); dev=ev.bool()
    if dev.sum()<2: return pmf.sum()*0.0
    B=pmf.shape[0]; ii=torch.randint(0,B,(npairs,),device=pmf.device); jj=torch.randint(0,B,(npairs,),device=pmf.device)
    v=dev[ii]&(bins[ii]<bins[jj])
    if v.sum()==0: return pmf.sum()*0.0
    ii,jj=ii[v],jj[v]; ti=bins[ii].clamp(0,pmf.shape[1]-1)
    return torch.exp(-(cif[ii,ti]-cif[jj,ti])/sigma).mean()

def pp_nll(lam, dt, obs):
    # exponential TPP next-arrival NLL: observed -> -log lam + lam*dt ; censored -> lam*dt
    lam=lam.clamp(1e-6,1e3)
    return (-obs*torch.log(lam) + lam*dt).mean()

_BINS=np.arange(K,dtype=np.float32)
def risk_from_pmf(pmf): return -(pmf*_BINS).sum(1)

In [ ]:
# 5 · GradNorm-normalize -> PCGrad on the SHARED TRUNK only  
def pcgrad_backward(task_losses, shared_params, other_params):
    # per-task trunk gradients, unit-normalized (GradNorm proxy), then conflict-projected (PCGrad)
    flats=[]
    for Lk in task_losses:
        g=torch.autograd.grad(Lk, shared_params, retain_graph=True, allow_unused=True)
        g=[gi if gi is not None else torch.zeros_like(p) for gi,p in zip(g,shared_params)]
        flats.append(torch.cat([gi.reshape(-1) for gi in g]))
    flats=[f/(f.norm()+1e-8) for f in flats]                 # GradNorm-style scale normalization
    proj=[f.clone() for f in flats]
    for i in range(len(proj)):
        for j in range(len(flats)):
            if i==j: continue
            dot=(proj[i]*flats[j]).sum()
            if dot<0: proj[i]=proj[i]-dot/(flats[j].norm()**2+1e-8)*flats[j]
    combined=torch.stack(proj).sum(0)
    idx=0
    for p in shared_params:
        n=p.numel(); p.grad=combined[idx:idx+n].view_as(p).clone(); idx+=n
    # head / expert / sigma params: ordinary gradient of the summed (already-weighted) loss
    total=sum(task_losses)
    hg=torch.autograd.grad(total, other_params, retain_graph=False, allow_unused=True)
    for p,gi in zip(other_params,hg): p.grad=gi if gi is not None else None

In [ ]:
# 6 · build model + optimizer + Kendall uncertainty weighting
cards=meta['cat_cardinalities']; n_num=X_num.shape[1]; fnode=node_feat.shape[1]; d=64
trunk=Trunk(n_num,cards,fnode,d=d,p=0.2).to(device)
mmoe=MMoE(d,n_experts=4,n_tasks=4).to(device); heads=Heads(d,K).to(device)
log_sigma=nn.Parameter(torch.zeros(4,device=device))     # tasks: [survival, pp, closure, priority]
shared_params=list(trunk.parameters())
other_params=list(mmoe.parameters())+list(heads.parameters())+[log_sigma]
opt=torch.optim.Adam(shared_params+other_params, lr=1e-3, weight_decay=1e-5)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=8)

def forward(idx):
    all_vec=trunk.event_vectors(X_num,X_cat)
    return heads(mmoe(trunk(idx,all_vec)))

def weighted_tasks(idx):
    dlog,lam,clog,plog=forward(idx)
    logp=F.log_softmax(dlog,1)
    Lsv=deephit_nll(logp,bin_idx[idx],ev_t[idx])+0.5*rank_loss(logp.exp(),bin_idx[idx],ev_t[idx])
    Lpp=pp_nll(lam,dt_h[idx],tte_obs_t[idx])
    Lclo=F.binary_cross_entropy_with_logits(clog,road[idx].float())
    pm=prio[idx]>=0
    Lpri=F.binary_cross_entropy_with_logits(plog[pm],prio[idx][pm].float()) if pm.any() else Lsv*0
    raw=[Lsv,Lpp,Lclo,Lpri]
    # Kendall: 0.5*exp(-s)*L + 0.5*s
    return [0.5*torch.exp(-log_sigma[k])*raw[k]+0.5*log_sigma[k] for k in range(4)], raw

@torch.no_grad()
def predict(idx):
    trunk.eval(); mmoe.eval(); heads.eval()
    dlog,lam,clog,plog=forward(idx)
    return (F.softmax(dlog,1).cpu().numpy(), lam.cpu().numpy(),
            torch.sigmoid(clog).cpu().numpy(), torch.sigmoid(plog).cpu().numpy())

def val_cindex():
    pmf,_,_,_=predict(va_idx)
    return concordance_index_censored(va_e.astype(bool),va_t,risk_from_pmf(pmf))[0]

In [ ]:
# 7 · train (PCGrad surgery on the shared trunk; select on val DeepHit C-index)
bs, best_c, best_state, patience = 1024, -1.0, None, 0
for epoch in range(180):
    trunk.train(); mmoe.train(); heads.train()
    perm=tr_idx[torch.randperm(tr_idx.numel(),device=device)]
    for s0 in range(0,perm.numel(),bs):
        b=perm[s0:s0+bs]; opt.zero_grad()
        wtasks,_=weighted_tasks(b)
        pcgrad_backward(wtasks, shared_params, other_params)
        torch.nn.utils.clip_grad_norm_(shared_params+other_params,5.0); opt.step()
    vc=val_cindex(); sched.step(vc)
    if vc>best_c+1e-4:
        best_c=vc; patience=0
        best_state={'t':trunk.state_dict(),'m':mmoe.state_dict(),'h':heads.state_dict(),'s':log_sigma.detach().clone()}
    else: patience+=1
    if epoch%5==0:
        with torch.no_grad(): _,raw=weighted_tasks(va_idx)
        print('epoch %3d val_C %.4f | Lsv %.3f Lpp %.3f Lclo %.3f Lpri %.3f | sigma %s'%(
            epoch,vc,raw[0],raw[1],raw[2],raw[3],np.round(torch.exp(log_sigma).detach().cpu().numpy(),2)))
    if patience>=30: print('early stop @',epoch); break
trunk.load_state_dict(best_state['t']); mmoe.load_state_dict(best_state['m']); heads.load_state_dict(best_state['h'])
with torch.no_grad(): log_sigma.copy_(best_state['s'])
print('restored best val C-index =', round(best_c,4))

In [ ]:
# 8 · evaluate both heads
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw, integrated_brier_score
from sklearn.metrics import roc_auc_score
pmf_te,lam_te,clo_te,pri_te=predict(te_idx)

def time_grid(tt,ee,tr_t,n=20):
    ev=tt[ee.astype(bool)]; lo,hi=np.percentile(ev,5),np.percentile(ev,95)
    return np.linspace(max(lo,min(tt.min(),tr_t.min())+1e-3),min(hi,min(tt.max(),tr_t.max())-1e-3),n)
def surv_at(pmf,times):
    cum=np.cumsum(pmf,1); right=cuts[1:]; S=np.ones((pmf.shape[0],len(times)))
    for j,t in enumerate(times):
        k=np.searchsorted(right,t,side='right')
        if k>0: S[:,j]=1.0-cum[:,min(k-1,pmf.shape[1]-1)]
    return S
tr_t,tr_e=durations[split==0],events[split==0]; te_t,te_e=durations[split==2],events[split==2]
grid=time_grid(te_t,te_e,tr_t)
y_tr=Surv.from_arrays(tr_e.astype(bool),np.clip(tr_t,1,None)); y_te=Surv.from_arrays(te_e.astype(bool),np.clip(te_t,1,None))
risk=risk_from_pmf(pmf_te)
cH=concordance_index_censored(te_e.astype(bool),te_t,risk)[0]
cI=concordance_index_ipcw(y_tr,y_te,risk,tau=float(grid.max()))[0]
ibs=integrated_brier_score(y_tr,y_te,surv_at(pmf_te,grid),grid)
print('SURVIVAL head :  C-Harrell %.4f  C-IPCW %.4f  IBS %.4f'%(cH,cI,ibs))
print('Tier0 baseline:  C-Harrell 0.5917  C-IPCW 0.5931  IBS 0.0596  (should not regress)')
# Hawkes head: does intensity rank next-incident timing?  (higher lambda -> sooner)
tte_te=z['tte_min'][split==2]; obs_te=z['tte_observed'][split==2]
cH_pp=concordance_index_censored(obs_te.astype(bool),tte_te,lam_te)[0]
print('HAWKES head   :  next-incident C-index %.4f  (mean intensity %.4f /h)'%(cH_pp,float(lam_te.mean())))
print('aux: closure AUC %.3f'%roc_auc_score(road[te_idx].cpu().numpy(),clo_te))

In [ ]:
# 9 · save weights + predictions (incl. per-junction next-incident intensity)
os.makedirs('out',exist_ok=True)
torch.save({'trunk':trunk.state_dict(),'mmoe':mmoe.state_dict(),'heads':heads.state_dict(),
            'log_sigma':log_sigma.detach().cpu(),'cuts':cuts,
            'config':{'d':d,'K':int(K),'cards':cards,'n_num':int(n_num),'fnode':int(fnode),'L':int(L)}},
           'out/trunk_mtl.pt')
pmf_all,lam_all,clo_all,pri_all=predict(torch.arange(N,device=device))
# aggregate predicted intensity per junction (predictive hotspot map)
node_all=z['node_id']; node_intensity=np.zeros(V)
for v in range(V):
    m=node_all==v
    if m.any(): node_intensity[v]=float(lam_all[m].mean())
np.savez_compressed('out/preds_mtl.npz',
    pmf=pmf_all, intensity=lam_all, closure_prob=clo_all, priority_prob=pri_all,
    node_intensity=node_intensity, cuts=cuts, event_id=z['event_id'], node_id=node_all,
    split=split, durations=durations, events=events, tte_min=z['tte_min'], tte_observed=z['tte_observed'],
    road_closure=z['road_closure'], priority_high=z['priority_high'])
print('saved out/trunk_mtl.pt , out/preds_mtl.npz')
try:
    from google.colab import files
    files.download('out/trunk_mtl.pt'); files.download('out/preds_mtl.npz')
except Exception as e: print('(not on Colab; files in ./out)', e)